In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
np.set_printoptions(suppress=True)

In [12]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM, RDM, CDM
from adapters import Adapter, TorchAdapter

# Transformer

In [13]:
class MAB(nn.Module):
    """
    Multihead Attention Block using torch.nn.MultiheadAttention.

    Args:
        dim_Q:  feature dim of Q input  (B, T_q, dim_Q)
        dim_K:  feature dim of K input  (B, T_k, dim_K)
        dim_V:  internal/embed dim for attention and output
        num_heads: number of attention heads
        ln: whether to use LayerNorm before/after the FFN residual
    """
    def __init__(self, query_dim: int, key_dim: int = None, num_heads: int = 4, ln: bool = False, dropout: float = 0.0):
        super().__init__()

        if key_dim is None:
            key_dim = query_dim

        # Built-in MHA; batch_first=True to accept (B, T, C)
        self.attn = nn.MultiheadAttention(
            embed_dim=query_dim,
            kdim=key_dim,
            vdim=key_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.ln0 = nn.LayerNorm(query_dim) if ln else None
        self.ln1 = nn.LayerNorm(query_dim) if ln else None

        self.fc_o = nn.Linear(query_dim, query_dim)

    def forward(self,
        query: torch.Tensor,
        key: torch.Tensor,
        attn_mask: torch.Tensor = None,
        key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Q: (B, T_q, dim_Q)
        K: (B, T_k, dim_K) — used for both keys and values
        attn_mask: optional (T_q, T_k) or (B*num_heads, T_q, T_k)
        key_padding_mask: optional (B, T_k), True for PAD positions
        """

        attn_out, _ = self.attn(
            query=query,
            key=key,
            value=key,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )

        # 3) First residual (q + attention), optional LayerNorm
        out = query + attn_out
        if self.ln0 is not None:
            out = self.ln0(out)

        # 4) Simple FFN + residual (match original: ReLU then linear)
        out = out + F.gelu(self.fc_o(out))
        if self.ln1 is not None:
            out = self.ln1(out)

        return out


class SAB(nn.Module):
    def __init__(self, input_dim: int, num_heads: int = 4, ln: bool = False, dropout: float = 0.0):
        super().__init__()
        self.mab = MAB(query_dim=input_dim, num_heads=num_heads, ln=ln, dropout=dropout)

    def forward(self, X, attn_mask=None, key_padding_mask=None):
        return self.mab(X, X, attn_mask=attn_mask, key_padding_mask=key_padding_mask)

class PMA(nn.Module):
    def __init__(
        self,
        input_dim: int,
        seed_dim: int,
        num_heads: int = 4,
        num_seeds: int = 1,
        ln: bool = False,
        dropout: float = 0.0
):
        super().__init__()
        self.S = nn.Parameter(torch.Tensor(1, num_seeds, seed_dim))
        nn.init.xavier_uniform_(self.S)
        self.mab = MAB(query_dim=seed_dim, key_dim=input_dim, num_heads=num_heads, ln=ln, dropout=dropout)

    def forward(self, X):
        return self.mab(self.S.repeat(X.size(0), 1, 1), X)

In [14]:
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim: int,
        embed_dim: int = 64,
        seed_dim: int = 128,
        num_layers: int = 3,
        num_seeds: int = 1,
        num_heads: int = 4,
        ln: bool = False,
        dropout: float = 0.0,
        layer_dropout: float = 0.0,
    ):
        super().__init__()
        assert num_layers >= 1, "num_layers must be >= 1"

        self.input_embedding = nn.Linear(input_dim, embed_dim)

        self.layers = nn.ModuleList(
            [SAB(input_dim=embed_dim, num_heads=num_heads, ln=ln, dropout=dropout)
             for _ in range(num_layers)]
        )
        self.post_dropout = nn.Dropout(layer_dropout) if layer_dropout > 0 else nn.Identity()

        self.pma = PMA(
            input_dim=embed_dim,
            seed_dim=seed_dim,
            num_seeds=num_seeds,
            num_heads=num_heads,
            ln=ln,
            dropout=dropout
        )

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        """
        X: (B, T, C)
        attn_mask: optional (T, T) or (B*num_heads, T, T)
        key_padding_mask: optional (B, T) with True for PAD tokens
        """

        out = self.input_embedding(x)

        for sab in self.layers:
            out = sab(out, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
            out = self.post_dropout(out)

        pooled = self.pma(out)
        return pooled

In [15]:
class Decoder(nn.Module):

    def __init__(
        self,
        input_dim: int,
        proj_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        ln: bool = False,
        dropout: float = 0.0,
        layer_dropout: float = 0.0,
    ):
        super().__init__()
        assert num_layers >= 1, "num_layers must be >= 1"

        self.input_proj = nn.Linear(input_dim, proj_dim)

        self.layers = nn.ModuleList(
            [SAB(input_dim=proj_dim, num_heads=num_heads, ln=ln, dropout=dropout)
             for _ in range(num_layers)]
        )
        self.post_dropout = nn.Dropout(layer_dropout) if layer_dropout > 0 else nn.Identity()

        self.output_proj = nn.Linear(proj_dim, 2)


    def forward(self, x, attn_mask=None, key_padding_mask=None):
        """
        X: (B, T, C)
        attn_mask: optional (T, T) or (B*num_heads, T, T)
        key_padding_mask: optional (B, T) with True for PAD tokens
        """

        out = self.input_proj(x)

        for sab in self.layers:
            out = sab(out, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
            out = self.post_dropout(out)

        mu, log_var = self.output_proj(out).chunk(2, dim=-1)
        return mu, log_var

In [16]:
encoder = Encoder(num_layers=3, input_dim=64, seed_dim=128)
encoder.eval()
decoder = Decoder(input_dim=128 + 2, num_layers=3, proj_dim=64)
decoder.eval()

Decoder(
  (input_proj): Linear(in_features=130, out_features=64, bias=True)
  (layers): ModuleList(
    (0-2): 3 x SAB(
      (mab): MAB(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (fc_o): Linear(in_features=64, out_features=64, bias=True)
      )
    )
  )
  (post_dropout): Identity()
  (output_proj): Linear(in_features=64, out_features=2, bias=True)
)

In [17]:
batch_size = 8

data = torch.randn(batch_size, 400, 64)

params = torch.randn(batch_size, 180, 1)

intrinc_indices = torch.randn(batch_size, 180, 1)
regressor_indices = torch.randn(batch_size, 180, 1)

params_mask = torch.randint(0, 2, size=(batch_size, 180, 1)).to(torch.float32)

In [18]:
enc = encoder(data)# Simp

print(enc.shape)# lest BayesGPT

enc = enc.repeat(1, 180, 1)

decoder_inp = torch.cat((intrinc_indices, regressor_indices, enc), axis=-1)

print(decoder_inp.shape)

decoder_out = decoder(decoder_inp)

print(decoder_out[0].shape, decoder_out[1].shape)

torch.Size([8, 1, 128])
torch.Size([8, 180, 130])
torch.Size([8, 180, 1]) torch.Size([8, 180, 1])


In [19]:
torch.mean((decoder_out - params)**2 * params_mask)

TypeError: unsupported operand type(s) for -: 'tuple' and 'Tensor'

# Simulator using `NestedModelFamily`

We start with DDM

In [20]:
ddm_intrinsics = ["v", "a", "tau", "s_v", "s_tau", "decay"]

In [21]:
# TODO: (maybe) automatically detect which parameters have both intercepts and slopes as priors
ddm_priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

In [22]:
# TODO: Automatically generate ContextManager if none provided
context_manager = ContextManager()

In [23]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=ddm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [25]:
samples = model_family.batch_sample(
    batch_size=6,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    flatten_param_outputs=True
)

In [26]:
samples

{'model_names': ['DDM', 'DDM', 'DDM', 'DDM', 'DDM', 'DDM'],
 'design_configs': [{'1': ['v', 'a', 'tau', 's_v', 's_tau', 'decay'],
   'u_1': ['v', 'a', 'tau', 's_v'],
   'u_2': ['v', 'a', 'decay'],
   'u_3': ['a'],
   'u_4': ['v', 'a', 'tau'],
   'u_5': ['v', 'tau', 's_v', 'decay'],
   'u_6': ['v', 'tau', 's_v'],
   'u_7': ['v', 'a', 's_v']},
  {'1': ['v', 'a', 'tau', 's_v', 's_tau', 'decay'],
   'u_1': ['v', 'a', 'decay'],
   'u_2': ['v', 'a', 'decay'],
   'u_3': ['v', 's_v', 'decay']},
  {'1': ['v', 'a', 'tau', 's_v', 's_tau', 'decay'],
   'u_1': ['a', 'tau', 'decay'],
   'u_2': ['v', 's_v', 'decay'],
   'u_3': ['tau', 'decay'],
   'u_4': ['tau', 's_v'],
   'u_5': ['a', 's_v', 'decay'],
   'u_6': ['v', 'a', 'tau', 's_v'],
   'u_7': ['v', 'a', 'decay'],
   'u_8': ['v', 'tau', 'decay'],
   'u_9': ['a']},
  {'1': ['v', 'a', 'tau', 's_v', 's_tau', 'decay'],
   'u_1': ['v', 'a', 's_v'],
   'u_2': ['v', 'tau'],
   'u_3': ['a', 'tau', 's_v'],
   'u_4': ['a', 's_v'],
   'u_5': ['v', 'a', 'tau

In [27]:
adapter = Adapter()

In [30]:
samples["sim_data"]

{'rts': array([[0.62015241, 0.51627028, 1.86558652, ..., 0.53592116, 0.59880149,
         0.95358491],
        [1.69878829, 0.44187143, 0.43827716, ..., 0.97322954, 0.00010278,
         2.88761309],
        [0.44726434, 0.63079405, 0.78617489, ..., 0.19825918, 3.73926515,
         0.92458062],
        [1.56161988, 1.55470693, 2.97692537, ..., 4.09172569, 0.92458062,
         0.78117374],
        [6.98291349, 0.6496979 , 0.56858689, ..., 0.48595412, 0.29789431,
         0.39314966],
        [0.32123584, 0.29952088, 0.30618376, ..., 0.29789431, 0.39314966,
         0.27134975]]),
 'choices': array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
         1.        ],
        [0.        , 1.        , 1.        , ..., 0.47459133, 0.00010278,
         0.05732093],
        [1.        , 1.        , 1.        , ..., 0.19825918, 0.02405867,
         0.50533697],
        [0.        , 0.        , 0.        , ..., 0.01685157, 0.50533697,
         0.61224628],
        [0.        ,